# Komparasi Hybrid AutoEncoder dengan Repeated 5×5, CatBoost Multi-Seed, dan Stable Global OOF Threshold

Notebook ini membandingkan:

- AE + CatBoost (3-seed probability ensemble)
- AE + ExtraTrees
- AE + Random Forest
- AE + LightGBM

## Tujuan revisi

Versi ini difokuskan untuk meningkatkan **stabilitas performa dan threshold**, bukan sekadar mengejar satu skor final yang tinggi.

Strategi utama:

1. **Repeated Stratified 5-Fold × 5 repeat** sehingga terdapat 25 evaluasi outer validation.
2. CatBoost menggunakan ensemble tiga seed; probabilitas ketiga model dirata-ratakan.
3. Setiap sampel training memperoleh lima probabilitas out-of-fold (OOF), satu dari setiap repeat.
4. Probabilitas OOF setiap sampel dirata-ratakan untuk menghasilkan representasi probabilitas yang lebih stabil.
5. Untuk setiap fold evaluasi, threshold dicari dari seluruh sampel training **kecuali sampel fold tersebut**.
6. Threshold final adalah median dari 25 threshold cross-fitted OOF.
7. Threshold dipilih dengan memaksimalkan F1, dengan target utama:
   - Accuracy ≥ 95%
   - Recall fraud ≥ 85%
   - Precision fraud ≥ 10%
8. Model dipilih berdasarkan performa repeated CV dan kestabilannya, bukan dari final testing.
9. Final validation hanya mengonfirmasi fixed threshold; final testing hanya digunakan untuk estimasi akhir.

Metrik utama: Precision fraud, Recall/Detection Rate, F1 fraud, PR-AUC, ROC-AUC, Accuracy, serta standard deviation antar-fold.


In [ ]:
# Instalasi dependensi (jalankan sekali jika paket belum tersedia)
%pip install -q tensorflow catboost lightgbm imbalanced-learn scikit-learn matplotlib seaborn pandas numpy


In [ ]:
# ============================================================
# IMPORT LIBRARY + REPRODUCIBILITY
# ============================================================

import os
import gc
import random
import warnings
from pathlib import Path

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping


def set_all_seeds(seed):
    """Atur seluruh seed secara eksplisit untuk satu training run."""
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_all_seeds(RANDOM_STATE)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    warnings.warn(f"Deterministic TensorFlow ops tidak dapat diaktifkan: {exc}")

try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass

print("✅ Library berhasil dimuat.")
print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))


In [ ]:
# ============================================================
# KONFIGURASI UTAMA
# ============================================================

TRAIN_SIZE = 0.80
VAL_SIZE = 0.10
TEST_SIZE = 0.10

# Repeated Stratified K-Fold: 5 fold × 5 repeat = 25 evaluasi.
N_SPLITS = 5
N_REPEATS = 5
N_TOTAL_FOLDS = N_SPLITS * N_REPEATS

# Pembersihan exact duplicate sebelum split.
DROP_EXACT_DUPLICATES = True

# AutoEncoder
LATENT_DIM = 64
AE_HIDDEN_DIM = 128
AE_BATCH_SIZE = 64
AE_EPOCHS = 50
AE_PATIENCE = 5
HYBRID_BATCH_SIZE = 512

# SMOTE moderat.
SMOTE_SAMPLING_STRATEGY = 0.10
SMOTE_K_NEIGHBORS = 3

# CatBoost multi-seed probability ensemble.
# Seed aktual = base_seed + offset.
CATBOOST_SEED_OFFSETS = (0, 101, 202)
CATBOOST_ENSEMBLE_SIZE = len(CATBOOST_SEED_OFFSETS)

# Threshold constraints.
TARGET_MIN_ACCURACY = 0.95

# Target utama sesuai fokus stabilitas dan keseimbangan.
TARGET_MIN_RECALL_PRIMARY = 0.85
TARGET_MIN_PRECISION_PRIMARY = 0.10

# Fallback bila kombinasi utama tidak tersedia.
TARGET_MIN_RECALL_SECONDARY = 0.80
TARGET_MIN_PRECISION_SECONDARY = 0.05

assert np.isclose(TRAIN_SIZE + VAL_SIZE + TEST_SIZE, 1.0)
assert N_SPLITS >= 2
assert N_REPEATS >= 2
assert CATBOOST_ENSEMBLE_SIZE >= 2
assert 0 < SMOTE_SAMPLING_STRATEGY <= 1
assert 0 <= TARGET_MIN_PRECISION_SECONDARY <= TARGET_MIN_PRECISION_PRIMARY <= 1
assert 0 <= TARGET_MIN_RECALL_SECONDARY <= TARGET_MIN_RECALL_PRIMARY <= 1

print("✅ Konfigurasi valid.")
print(f"Split utama               : {TRAIN_SIZE:.0%}/{VAL_SIZE:.0%}/{TEST_SIZE:.0%}")
print(
    f"Repeated CV               : {N_SPLITS}-fold × "
    f"{N_REPEATS} repeat = {N_TOTAL_FOLDS} evaluasi"
)
print(f"Hybrid feature dimension  : 29 + {LATENT_DIM} + 1 = {29 + LATENT_DIM + 1}")
print(f"SMOTE sampling strategy   : {SMOTE_SAMPLING_STRATEGY}")
print(f"CatBoost ensemble         : {CATBOOST_ENSEMBLE_SIZE} seed")
print(
    "Primary threshold target : "
    f"Acc≥{TARGET_MIN_ACCURACY:.0%}, "
    f"Recall≥{TARGET_MIN_RECALL_PRIMARY:.0%}, "
    f"Precision≥{TARGET_MIN_PRECISION_PRIMARY:.0%}"
)
print(
    "Secondary target         : "
    f"Acc≥{TARGET_MIN_ACCURACY:.0%}, "
    f"Recall≥{TARGET_MIN_RECALL_SECONDARY:.0%}, "
    f"Precision≥{TARGET_MIN_PRECISION_SECONDARY:.0%}"
)
print(
    "Estimasi training         : "
    f"{N_TOTAL_FOLDS} AutoEncoder dan "
    f"{N_TOTAL_FOLDS * (CATBOOST_ENSEMBLE_SIZE + 3)} classifier"
)


In [ ]:
# ============================================================
# LOAD, VALIDASI, DAN PEMBERSIHAN DATASET
# ============================================================

DATA_PATH = Path("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")

if not DATA_PATH.exists() and Path("/content/creditcard.csv").exists():
    DATA_PATH = Path("/content/creditcard.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "File creditcard.csv tidak ditemukan. Letakkan file tersebut pada folder "
        "yang sama dengan notebook atau di /content/creditcard.csv."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"Class", "Time", "Amount"} | {f"V{i}" for i in range(1, 29)}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

if not set(df["Class"].dropna().unique()).issubset({0, 1}):
    raise ValueError("Kolom Class harus berupa label biner 0 dan 1.")

original_shape = df.shape
duplicate_count = int(df.duplicated().sum())

if DROP_EXACT_DUPLICATES and duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Dataset path              : {DATA_PATH.resolve()}")
print(f"Shape awal                : {original_shape}")
print(f"Exact duplicate ditemukan : {duplicate_count:,}")
print(f"Shape setelah pembersihan : {df.shape}")
print(f"Missing value             : {df.isna().sum().sum():,}")

if duplicate_count > 0 and not DROP_EXACT_DUPLICATES:
    warnings.warn(
        "Exact duplicate belum dihapus. Sampel identik berisiko tersebar "
        "ke training dan testing."
    )

df.head()


In [ ]:
# ============================================================
# EDA SINGKAT
# ============================================================

class_counts = df["Class"].value_counts().sort_index()
normal_count = int(class_counts.get(0, 0))
fraud_count = int(class_counts.get(1, 0))

print(f"Transaksi Sah (0)      : {normal_count:,} ({normal_count / len(df) * 100:.4f}%)")
print(f"Transaksi Penipuan (1) : {fraud_count:,} ({fraud_count / len(df) * 100:.4f}%)")
print(f"Imbalance ratio        : 1 fraud : {normal_count / fraud_count:.2f} normal")

plt.figure(figsize=(7, 4))
ax = sns.barplot(
    x=["Sah (0)", "Penipuan (1)"],
    y=[normal_count, fraud_count],
)
for i, value in enumerate([normal_count, fraud_count]):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom", fontweight="bold")
plt.title("Distribusi Kelas Credit Card Fraud")
plt.ylabel("Jumlah Transaksi")
plt.tight_layout()
plt.savefig("distribusi_kelas_creditcard.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# FEATURE-TARGET PREPARATION
# ============================================================
# Konfigurasi 29 fitur: V1-V28 + Amount. Time dihapus.
# Imputasi dan scaling dilakukan di dalam fold agar leakage-free.

feature_columns = [f"V{i}" for i in range(1, 29)] + ["Amount"]

X = df[feature_columns].to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int8)

print("Shape X       :", X.shape)
print("Shape y       :", y.shape)
print("Jumlah fitur  :", X.shape[1])
print("Jumlah fraud  :", int((y == 1).sum()))
print("Jumlah normal :", int((y == 0).sum()))


## Fungsi bantu

Bagian berikut mendefinisikan:

- preprocessing leakage-free;
- AutoEncoder yang hanya belajar dari transaksi sah pada training fold;
- ekstraksi fitur hybrid;
- SMOTE khusus training fold;
- threshold berbasis F1 tanpa F2;
- CatBoost probability ensemble tiga seed;
- evaluasi binary classification.

Semua seed berubah secara deterministik pada setiap fold sehingga eksperimen tetap reproducible, tetapi stabilitas terhadap variasi fold dan seed masih dapat diukur.


In [ ]:
# ============================================================
# PREPROCESSING HELPERS
# ============================================================

def fit_preprocessor(X_train_raw):
    """Fit median imputer dan StandardScaler hanya pada training data."""
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_imputed = imputer.fit_transform(X_train_raw)
    X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)

    return imputer, scaler, X_train_scaled


def transform_preprocessor(imputer, scaler, X_raw):
    """Transform data lain memakai parameter dari training data."""
    X_imputed = imputer.transform(X_raw)
    return scaler.transform(X_imputed).astype(np.float32)


In [ ]:
# ============================================================
# AUTOENCODER
# ============================================================

def build_autoencoder(input_dim, latent_dim=LATENT_DIM, hidden_dim=AE_HIDDEN_DIM):
    """AutoEncoder: input -> hidden -> latent -> hidden -> reconstruction."""
    inp = Input(shape=(input_dim,), name="input_features")

    hidden_enc = Dense(
        hidden_dim,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="encoder_hidden",
    )(inp)

    latent = Dense(
        latent_dim,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="latent_space",
    )(hidden_enc)

    hidden_dec = Dense(
        hidden_dim,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="decoder_hidden",
    )(latent)

    out = Dense(input_dim, activation="linear", name="reconstruction")(hidden_dec)

    autoencoder = Model(inp, out, name="fraud_autoencoder")
    encoder = Model(inp, latent, name="fraud_encoder")

    autoencoder.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="mse",
    )
    return autoencoder, encoder


def fit_autoencoder_on_normal(X_normal_train, seed=RANDOM_STATE):
    """Fit AutoEncoder hanya pada transaksi normal dari training fold."""
    if len(X_normal_train) < 20:
        raise ValueError("Jumlah transaksi normal terlalu sedikit untuk AutoEncoder.")

    tf.keras.backend.clear_session()
    set_all_seeds(seed)

    X_ae_train, X_ae_val = train_test_split(
        X_normal_train,
        test_size=0.10,
        random_state=seed,
        shuffle=True,
    )

    autoencoder, encoder = build_autoencoder(X_normal_train.shape[1])

    history = autoencoder.fit(
        X_ae_train,
        X_ae_train,
        validation_data=(X_ae_val, X_ae_val),
        epochs=AE_EPOCHS,
        batch_size=AE_BATCH_SIZE,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=AE_PATIENCE,
                restore_best_weights=True,
            )
        ],
        verbose=0,
        shuffle=True,
    )

    return autoencoder, encoder, history


In [ ]:
# ============================================================
# HYBRID FEATURE EXTRACTION
# ============================================================

def transform_hybrid_features(
    autoencoder,
    encoder,
    X_scaled,
    batch_size=HYBRID_BATCH_SIZE,
):
    """
    Hybrid = standardized original features + latent features
             + per-sample reconstruction MSE.
    """
    X_scaled = np.asarray(X_scaled, dtype=np.float32)

    n_samples, n_original = X_scaled.shape
    n_latent = int(encoder.output_shape[-1])
    X_hybrid = np.empty((n_samples, n_original + n_latent + 1), dtype=np.float32)

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        X_batch = X_scaled[start:end]

        latent_batch = encoder.predict(
            X_batch,
            batch_size=batch_size,
            verbose=0,
        ).astype(np.float32, copy=False)

        reconstruction_batch = autoencoder.predict(
            X_batch,
            batch_size=batch_size,
            verbose=0,
        ).astype(np.float32, copy=False)

        reconstruction_error = np.mean(
            np.square(X_batch - reconstruction_batch),
            axis=1,
            keepdims=True,
        ).astype(np.float32, copy=False)

        X_hybrid[start:end] = np.concatenate(
            [X_batch, latent_batch, reconstruction_error],
            axis=1,
        )

    return X_hybrid


In [ ]:
# ============================================================
# SMOTE
# ============================================================

def apply_smote(
    X_train_hybrid,
    y_train,
    label,
    sampling_strategy=SMOTE_SAMPLING_STRATEGY,
    seed=RANDOM_STATE,
):
    """SMOTE hanya diterapkan pada training fold."""
    minority_count = int((y_train == 1).sum())
    effective_k = min(SMOTE_K_NEIGHBORS, minority_count - 1)

    if effective_k < 1:
        raise ValueError(f"[{label}] Sampel fraud terlalu sedikit untuk SMOTE.")

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=seed,
        k_neighbors=effective_k,
    )

    X_resampled, y_resampled = smote.fit_resample(X_train_hybrid, y_train)
    X_resampled = X_resampled.astype(np.float32, copy=False)

    print(
        f"[{label}] SMOTE | fraud {minority_count:,} -> "
        f"{int((y_resampled == 1).sum()):,}; "
        f"total {len(y_train):,} -> {len(y_resampled):,}"
    )
    return X_resampled, y_resampled


In [ ]:
# ============================================================
# BALANCED F1 THRESHOLD SELECTION — TANPA F2
# ============================================================

def find_balanced_f1_threshold(
    y_true,
    y_prob,
    min_accuracy=TARGET_MIN_ACCURACY,
    min_recall_primary=TARGET_MIN_RECALL_PRIMARY,
    min_precision_primary=TARGET_MIN_PRECISION_PRIMARY,
    min_recall_secondary=TARGET_MIN_RECALL_SECONDARY,
    min_precision_secondary=TARGET_MIN_PRECISION_SECONDARY,
):
    """
    Cari threshold pada validation/tuning data, bukan final testing data.

    Hierarki:
    1. Accuracy, recall, dan precision memenuhi target primary;
       pilih threshold dengan F1 tertinggi.
    2. Jika tidak tersedia, gunakan target secondary;
       pilih threshold dengan F1 tertinggi.
    3. Jika tidak tersedia, pertahankan accuracy minimum;
       pilih F1 tertinggi.
    4. Jika accuracy minimum juga tidak tercapai;
       pilih F1 tertinggi secara keseluruhan.

    F2 sengaja tidak digunakan.
    """
    y_true = np.asarray(y_true, dtype=np.int8)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    if y_true.ndim != 1 or y_prob.ndim != 1 or len(y_true) != len(y_prob):
        raise ValueError("y_true dan y_prob harus berupa vector dengan panjang sama.")
    if set(np.unique(y_true)) - {0, 1}:
        raise ValueError("y_true harus biner.")
    if len(np.unique(y_true)) < 2:
        raise ValueError("Threshold tuning membutuhkan kedua kelas.")
    if not np.all(np.isfinite(y_prob)):
        raise ValueError("y_prob mengandung NaN atau infinity.")

    # Gabungkan grid tetap dan quantile probabilitas agar threshold rendah/tinggi
    # tetap dapat dievaluasi tanpa bergantung hanya pada rentang linear.
    fixed_grid = np.linspace(0.0001, 0.9999, 1000)
    quantile_grid = np.quantile(y_prob, np.linspace(0.0, 1.0, 1001))
    thresholds = np.unique(
        np.clip(
            np.concatenate([fixed_grid, quantile_grid, np.array([0.5])]),
            1e-8,
            1.0 - 1e-8,
        )
    )

    rows = []

    for threshold in thresholds:
        pred = (y_prob >= threshold).astype(np.int8)
        tn, fp, fn, tp = confusion_matrix(
            y_true, pred, labels=[0, 1]
        ).ravel()

        accuracy = accuracy_score(y_true, pred)
        recall = recall_score(
            y_true, pred, pos_label=1, zero_division=0
        )
        precision = precision_score(
            y_true, pred, pos_label=1, zero_division=0
        )
        f1 = f1_score(
            y_true, pred, pos_label=1, zero_division=0
        )
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        rows.append(
            {
                "threshold": float(threshold),
                "accuracy": accuracy,
                "recall": recall,
                "precision": precision,
                "f1": f1,
                "fpr": fpr,
                "tp": int(tp),
                "fp": int(fp),
                "fn": int(fn),
                "tn": int(tn),
            }
        )

    table = pd.DataFrame(rows)

    primary = table[
        (table["accuracy"] >= min_accuracy)
        & (table["recall"] >= min_recall_primary)
        & (table["precision"] >= min_precision_primary)
    ]

    secondary = table[
        (table["accuracy"] >= min_accuracy)
        & (table["recall"] >= min_recall_secondary)
        & (table["precision"] >= min_precision_secondary)
    ]

    accuracy_safe = table[table["accuracy"] >= min_accuracy]

    sort_columns = ["f1", "precision", "recall", "accuracy", "fpr", "threshold"]
    sort_ascending = [False, False, False, False, True, False]

    if not primary.empty:
        best = primary.sort_values(
            by=sort_columns,
            ascending=sort_ascending,
        ).iloc[0]
        status = "PRIMARY_BALANCED_F1"

    elif not secondary.empty:
        best = secondary.sort_values(
            by=sort_columns,
            ascending=sort_ascending,
        ).iloc[0]
        status = "SECONDARY_BALANCED_F1"

    elif not accuracy_safe.empty:
        best = accuracy_safe.sort_values(
            by=sort_columns,
            ascending=sort_ascending,
        ).iloc[0]
        status = "ACCURACY_SAFE_MAX_F1"

    else:
        best = table.sort_values(
            by=sort_columns,
            ascending=sort_ascending,
        ).iloc[0]
        status = "GLOBAL_MAX_F1"

    return float(best["threshold"]), best.to_dict(), table, status


In [ ]:
# ============================================================
# MODEL FACTORY, CATBOOST MULTI-SEED ENSEMBLE, DAN METRIK
# ============================================================

CATBOOST_ENSEMBLE_NAME = "AE + CatBoost (3-Seed Ensemble)"

MODEL_NAMES = [
    CATBOOST_ENSEMBLE_NAME,
    "AE + ExtraTrees",
    "AE + Random Forest",
    "AE + LightGBM",
]


def make_single_model(model_name, seed):
    """Buat satu estimator berdasarkan nama model dan seed."""
    if model_name == CATBOOST_ENSEMBLE_NAME:
        return CatBoostClassifier(
            iterations=350,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            random_seed=seed,
            eval_metric="AUC",
            verbose=0,
            allow_writing_files=False,
            thread_count=1,
        )

    if model_name == "AE + ExtraTrees":
        return ExtraTreesClassifier(
            n_estimators=350,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=1,
            random_state=seed,
            class_weight=None,
        )

    if model_name == "AE + Random Forest":
        return RandomForestClassifier(
            n_estimators=350,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=1,
            random_state=seed,
            class_weight=None,
        )

    if model_name == "AE + LightGBM":
        return LGBMClassifier(
            n_estimators=350,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            objective="binary",
            random_state=seed,
            n_jobs=1,
            deterministic=True,
            force_col_wise=True,
            verbosity=-1,
            bagging_seed=seed,
            feature_fraction_seed=seed,
            data_random_seed=seed,
        )

    raise ValueError(f"Model tidak dikenali: {model_name}")


def build_model_bundle(model_name, base_seed):
    """
    CatBoost menghasilkan tiga estimator dengan seed berbeda.
    Model lain menghasilkan satu estimator.
    """
    if model_name == CATBOOST_ENSEMBLE_NAME:
        return [
            make_single_model(model_name, base_seed + offset)
            for offset in CATBOOST_SEED_OFFSETS
        ]

    return [make_single_model(model_name, base_seed)]


def fit_model_bundle(model_bundle, X_train, y_train):
    """Fit seluruh estimator di dalam bundle."""
    for estimator in model_bundle:
        estimator.fit(X_train, y_train)
    return model_bundle


def predict_model_bundle(model_bundle, X_eval):
    """Rata-ratakan probabilitas jika bundle berisi lebih dari satu model."""
    probabilities = np.column_stack(
        [
            estimator.predict_proba(X_eval)[:, 1]
            for estimator in model_bundle
        ]
    )
    return probabilities.mean(axis=1)


def compute_binary_metrics(y_true, y_pred, y_prob):
    """Metrik utama dan confusion-matrix counts."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision Fraud (%)": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ) * 100,
        "Detection Rate (%)": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ) * 100,
        "F1 Fraud (%)": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ) * 100,
        "ROC-AUC (%)": roc_auc_score(y_true, y_prob) * 100,
        "PR-AUC (%)": average_precision_score(y_true, y_prob) * 100,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }


def threshold_confirmation_status(metrics):
    """Konfirmasi fixed OOF threshold pada validation holdout."""
    accuracy = metrics["Accuracy (%)"] / 100
    precision = metrics["Precision Fraud (%)"] / 100
    recall = metrics["Detection Rate (%)"] / 100

    if (
        accuracy >= TARGET_MIN_ACCURACY
        and recall >= TARGET_MIN_RECALL_PRIMARY
        and precision >= TARGET_MIN_PRECISION_PRIMARY
    ):
        return "PRIMARY_CONFIRMED"

    if (
        accuracy >= TARGET_MIN_ACCURACY
        and recall >= TARGET_MIN_RECALL_SECONDARY
        and precision >= TARGET_MIN_PRECISION_SECONDARY
    ):
        return "SECONDARY_CONFIRMED"

    return "VALIDATION_WARNING"


In [ ]:
# ============================================================
# SPLIT UTAMA 80:10:10
# ============================================================

X_train_pool, X_temp, y_train_pool, y_temp = train_test_split(
    X,
    y,
    test_size=VAL_SIZE + TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
X_val_raw, X_test_raw, y_val_holdout, y_test_holdout = train_test_split(
    X_temp,
    y_temp,
    test_size=relative_test_size,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

split_audit = pd.DataFrame(
    [
        {
            "Split": "Training pool",
            "Samples": len(y_train_pool),
            "Fraud": int((y_train_pool == 1).sum()),
            "Fraud rate (%)": (y_train_pool == 1).mean() * 100,
        },
        {
            "Split": "Validation holdout",
            "Samples": len(y_val_holdout),
            "Fraud": int((y_val_holdout == 1).sum()),
            "Fraud rate (%)": (y_val_holdout == 1).mean() * 100,
        },
        {
            "Split": "Final testing holdout",
            "Samples": len(y_test_holdout),
            "Fraud": int((y_test_holdout == 1).sum()),
            "Fraud rate (%)": (y_test_holdout == 1).mean() * 100,
        },
    ]
)

split_audit


## Repeated 5×5 OOF dan Cross-Fitted Global Threshold

Prosedur:

1. Training pool 80% dibagi dengan Repeated Stratified 5-Fold × 5.
2. Setiap fold melatih preprocessing, AutoEncoder, SMOTE, dan classifier hanya dari outer-training.
3. CatBoost pada setiap fold terdiri atas tiga model dengan seed berbeda; probabilitasnya dirata-ratakan.
4. Setiap sampel memperoleh lima probabilitas OOF, kemudian dirata-ratakan.
5. Untuk mengevaluasi satu fold, threshold dicari menggunakan rata-rata OOF seluruh sampel **selain sampel fold evaluasi**.
6. Prosedur tersebut menghasilkan 25 threshold cross-fitted yang leakage-safe.
7. Median dari 25 threshold digunakan sebagai fixed global threshold untuk final training.
8. Standard deviation dan IQR threshold digunakan untuk mengukur stabilitas threshold.


In [ ]:
# ============================================================
# REPEATED 5×5 OOF PROBABILITIES
# ============================================================

repeated_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

n_train_samples = len(y_train_pool)

# Shape: repeat × sample.
oof_probabilities = {
    model_name: np.full(
        (N_REPEATS, n_train_samples),
        np.nan,
        dtype=np.float32,
    )
    for model_name in MODEL_NAMES
}

oof_fold_ids = np.full(
    (N_REPEATS, n_train_samples),
    -1,
    dtype=np.int16,
)

ranking_fold_records = []

for split_number, (outer_train_idx, outer_val_idx) in enumerate(
    repeated_cv.split(X_train_pool, y_train_pool),
    start=1,
):
    repeat_num = ((split_number - 1) // N_SPLITS) + 1
    fold_num = ((split_number - 1) % N_SPLITS) + 1
    repeat_idx = repeat_num - 1

    fold_seed = RANDOM_STATE + (repeat_num * 1000) + fold_num

    print("\n" + "=" * 100)
    print(
        f"REPEAT {repeat_num}/{N_REPEATS} | "
        f"FOLD {fold_num}/{N_SPLITS} | "
        f"RUN {split_number}/{N_TOTAL_FOLDS}"
    )
    print("=" * 100)

    X_outer_train_raw = X_train_pool[outer_train_idx]
    y_outer_train = y_train_pool[outer_train_idx]
    X_outer_val_raw = X_train_pool[outer_val_idx]
    y_outer_val = y_train_pool[outer_val_idx]

    print(
        f"Outer train : {X_outer_train_raw.shape} | "
        f"fraud={int((y_outer_train == 1).sum())}"
    )
    print(
        f"Outer val   : {X_outer_val_raw.shape} | "
        f"fraud={int((y_outer_val == 1).sum())}"
    )

    imputer, scaler, X_outer_train_scaled = fit_preprocessor(
        X_outer_train_raw
    )
    X_outer_val_scaled = transform_preprocessor(
        imputer,
        scaler,
        X_outer_val_raw,
    )

    X_normal_train = X_outer_train_scaled[y_outer_train == 0]
    autoencoder, encoder, ae_history = fit_autoencoder_on_normal(
        X_normal_train,
        seed=fold_seed,
    )

    X_outer_train_hybrid = transform_hybrid_features(
        autoencoder,
        encoder,
        X_outer_train_scaled,
    )
    X_outer_val_hybrid = transform_hybrid_features(
        autoencoder,
        encoder,
        X_outer_val_scaled,
    )

    X_train_smote, y_train_smote = apply_smote(
        X_outer_train_hybrid,
        y_outer_train,
        label=f"Repeat {repeat_num} Fold {fold_num}",
        seed=fold_seed,
    )

    oof_fold_ids[repeat_idx, outer_val_idx] = fold_num

    for model_name in MODEL_NAMES:
        model_bundle = build_model_bundle(
            model_name,
            base_seed=fold_seed,
        )

        print(
            f"[Repeat {repeat_num} Fold {fold_num}] "
            f"Training {model_name} "
            f"({len(model_bundle)} estimator)"
        )

        fit_model_bundle(
            model_bundle,
            X_train_smote,
            y_train_smote,
        )

        prob_outer_val = predict_model_bundle(
            model_bundle,
            X_outer_val_hybrid,
        ).astype(np.float32)

        oof_probabilities[model_name][
            repeat_idx,
            outer_val_idx,
        ] = prob_outer_val

        ranking_fold_records.append(
            {
                "Repeat": repeat_num,
                "Fold": fold_num,
                "Model": model_name,
                "Estimator Count": len(model_bundle),
                "ROC-AUC (%)": (
                    roc_auc_score(y_outer_val, prob_outer_val) * 100
                ),
                "PR-AUC (%)": (
                    average_precision_score(
                        y_outer_val,
                        prob_outer_val,
                    ) * 100
                ),
            }
        )

        del model_bundle

    del (
        X_outer_train_raw,
        X_outer_val_raw,
        X_outer_train_scaled,
        X_outer_val_scaled,
        X_normal_train,
        X_outer_train_hybrid,
        X_outer_val_hybrid,
        X_train_smote,
        y_train_smote,
        autoencoder,
        encoder,
        ae_history,
        imputer,
        scaler,
    )

    gc.collect()
    tf.keras.backend.clear_session()

# Audit kelengkapan OOF.
if np.any(oof_fold_ids < 1):
    raise RuntimeError("Sebagian sampel belum memiliki fold OOF.")

for model_name in MODEL_NAMES:
    if np.isnan(oof_probabilities[model_name]).any():
        raise RuntimeError(
            f"OOF probabilities {model_name} masih mengandung NaN."
        )

df_ranking_folds = pd.DataFrame(ranking_fold_records)

print("\n✅ Seluruh probabilitas OOF berhasil dibuat.")
print(
    f"Total training: {N_TOTAL_FOLDS} AutoEncoder dan "
    f"{N_TOTAL_FOLDS * (CATBOOST_ENSEMBLE_SIZE + 3)} classifier."
)

# ============================================================
# AVERAGED OOF PROBABILITIES
# ============================================================

mean_oof_probabilities = {
    model_name: probabilities.mean(axis=0)
    for model_name, probabilities in oof_probabilities.items()
}

# ============================================================
# CROSS-FITTED THRESHOLD EVALUATION
# ============================================================
#
# Untuk setiap repeat-fold:
# - sampel fold tersebut tidak dipakai untuk mencari threshold;
# - threshold dicari dari averaged OOF probability sampel lain;
# - threshold kemudian diuji pada OOF probability fold evaluasi.
# ============================================================

crossfit_threshold_records = []
cv_metric_records = []

for model_name in MODEL_NAMES:
    averaged_prob = mean_oof_probabilities[model_name]

    for repeat_idx in range(N_REPEATS):
        repeat_num = repeat_idx + 1
        repeat_prob = oof_probabilities[model_name][repeat_idx]

        for fold_num in range(1, N_SPLITS + 1):
            evaluation_mask = oof_fold_ids[repeat_idx] == fold_num
            threshold_tuning_mask = ~evaluation_mask

            (
                threshold,
                threshold_info,
                _,
                threshold_status,
            ) = find_balanced_f1_threshold(
                y_train_pool[threshold_tuning_mask],
                averaged_prob[threshold_tuning_mask],
            )

            y_fold = y_train_pool[evaluation_mask]
            prob_fold = repeat_prob[evaluation_mask]
            pred_fold = (
                prob_fold >= threshold
            ).astype(np.int8)

            metrics = compute_binary_metrics(
                y_fold,
                pred_fold,
                prob_fold,
            )

            crossfit_threshold_records.append(
                {
                    "Repeat": repeat_num,
                    "Fold": fold_num,
                    "Model": model_name,
                    "Threshold": threshold,
                    "Threshold Status": threshold_status,
                    "Tuning Samples": int(threshold_tuning_mask.sum()),
                    "Excluded Evaluation Samples": int(evaluation_mask.sum()),
                    "Tuning Accuracy (%)": threshold_info["accuracy"] * 100,
                    "Tuning Precision Fraud (%)": (
                        threshold_info["precision"] * 100
                    ),
                    "Tuning Detection Rate (%)": (
                        threshold_info["recall"] * 100
                    ),
                    "Tuning F1 Fraud (%)": threshold_info["f1"] * 100,
                }
            )

            cv_metric_records.append(
                {
                    "Repeat": repeat_num,
                    "Fold": fold_num,
                    "Model": model_name,
                    "Threshold": threshold,
                    "Threshold Source": (
                        "CROSSFIT_OOF_EXCLUDING_EVALUATION_FOLD"
                    ),
                    **metrics,
                }
            )

df_crossfit_thresholds = pd.DataFrame(
    crossfit_threshold_records
)
df_cv_folds = pd.DataFrame(cv_metric_records)

# ============================================================
# ROBUST GLOBAL THRESHOLD = MEDIAN 25 CROSS-FITTED THRESHOLDS
# ============================================================

threshold_summary_records = []
global_oof_reference_records = []

for model_name in MODEL_NAMES:
    model_thresholds = df_crossfit_thresholds.loc[
        df_crossfit_thresholds["Model"] == model_name,
        "Threshold",
    ].to_numpy(dtype=np.float64)

    robust_threshold = float(np.median(model_thresholds))
    q1 = float(np.quantile(model_thresholds, 0.25))
    q3 = float(np.quantile(model_thresholds, 0.75))

    threshold_summary_records.append(
        {
            "Model": model_name,
            "Robust Threshold": robust_threshold,
            "Threshold Mean": float(np.mean(model_thresholds)),
            "Threshold Std": float(
                np.std(model_thresholds, ddof=1)
            ),
            "Threshold Min": float(np.min(model_thresholds)),
            "Threshold Q1": q1,
            "Threshold Median": robust_threshold,
            "Threshold Q3": q3,
            "Threshold Max": float(np.max(model_thresholds)),
            "Threshold IQR": q3 - q1,
            "Threshold Count": len(model_thresholds),
        }
    )

    averaged_prob = mean_oof_probabilities[model_name]
    averaged_pred = (
        averaged_prob >= robust_threshold
    ).astype(np.int8)

    global_oof_reference_records.append(
        {
            "Model": model_name,
            "Threshold": robust_threshold,
            **compute_binary_metrics(
                y_train_pool,
                averaged_pred,
                averaged_prob,
            ),
        }
    )

df_threshold_summary = pd.DataFrame(
    threshold_summary_records
)
df_global_oof_reference = pd.DataFrame(
    global_oof_reference_records
)

print("\n" + "=" * 124)
print("STABILITAS 25 CROSS-FITTED OOF THRESHOLDS")
print("=" * 124)

for _, row in df_threshold_summary.iterrows():
    print(
        f"{row['Model']:<36} | "
        f"Median={row['Robust Threshold']:.6f} | "
        f"Std={row['Threshold Std']:.6f} | "
        f"IQR={row['Threshold IQR']:.6f} | "
        f"Range=[{row['Threshold Min']:.6f}, "
        f"{row['Threshold Max']:.6f}]"
    )

print(
    "\n✅ Cross-fitted threshold evaluation selesai. "
    "Sampel fold evaluasi tidak digunakan untuk mencari threshold."
)

df_cv_folds.head()


In [ ]:
# ============================================================
# RINGKASAN REPEATED 5×5 DAN PEMILIHAN MODEL STABIL
# ============================================================

metric_columns = [
    "Accuracy (%)",
    "Precision Fraud (%)",
    "Detection Rate (%)",
    "F1 Fraud (%)",
    "ROC-AUC (%)",
    "PR-AUC (%)",
]

summary_rows = []

for model_name in MODEL_NAMES:
    model_rows = df_cv_folds[
        df_cv_folds["Model"] == model_name
    ]
    threshold_row = df_threshold_summary[
        df_threshold_summary["Model"] == model_name
    ].iloc[0]

    row = {"Model": model_name}

    for metric in metric_columns:
        mean_value = model_rows[metric].mean()
        std_value = model_rows[metric].std(ddof=1)
        ci95 = 1.96 * std_value / np.sqrt(N_TOTAL_FOLDS)

        row[f"{metric} Mean"] = mean_value
        row[f"{metric} Std"] = std_value
        row[f"{metric} 95% CI"] = ci95
        row[f"{metric} Min"] = model_rows[metric].min()
        row[f"{metric} P10"] = model_rows[metric].quantile(0.10)

    row["Robust Threshold"] = threshold_row["Robust Threshold"]
    row["Threshold Std"] = threshold_row["Threshold Std"]
    row["Threshold IQR"] = threshold_row["Threshold IQR"]

    # Penalti terhadap variance.
    row["F1 Stability Score"] = (
        row["F1 Fraud (%) Mean"]
        - row["F1 Fraud (%) Std"]
    )
    row["Recall Stability Score"] = (
        row["Detection Rate (%) Mean"]
        - row["Detection Rate (%) Std"]
    )

    summary_rows.append(row)

df_cv_summary = pd.DataFrame(summary_rows)

print("=" * 156)
print(
    f"RINGKASAN REPEATED STRATIFIED "
    f"{N_SPLITS}-FOLD × {N_REPEATS} REPEAT"
)
print("=" * 156)

for _, row in df_cv_summary.iterrows():
    print(
        f"{row['Model']:<36} | "
        f"Acc {row['Accuracy (%) Mean']:.4f}±"
        f"{row['Accuracy (%) Std']:.4f} | "
        f"Prec {row['Precision Fraud (%) Mean']:.4f}±"
        f"{row['Precision Fraud (%) Std']:.4f} | "
        f"Recall {row['Detection Rate (%) Mean']:.4f}±"
        f"{row['Detection Rate (%) Std']:.4f} | "
        f"F1 {row['F1 Fraud (%) Mean']:.4f}±"
        f"{row['F1 Fraud (%) Std']:.4f} | "
        f"PR-AUC {row['PR-AUC (%) Mean']:.4f}±"
        f"{row['PR-AUC (%) Std']:.4f} | "
        f"Thr {row['Robust Threshold']:.6f}"
    )

# ============================================================
# MODEL SELECTION
# ============================================================

primary_eligible = df_cv_summary[
    (df_cv_summary["Accuracy (%) Mean"] >= TARGET_MIN_ACCURACY * 100)
    & (
        df_cv_summary["Detection Rate (%) Mean"]
        >= TARGET_MIN_RECALL_PRIMARY * 100
    )
    & (
        df_cv_summary["Precision Fraud (%) Mean"]
        >= TARGET_MIN_PRECISION_PRIMARY * 100
    )
].copy()

secondary_eligible = df_cv_summary[
    (df_cv_summary["Accuracy (%) Mean"] >= TARGET_MIN_ACCURACY * 100)
    & (
        df_cv_summary["Detection Rate (%) Mean"]
        >= TARGET_MIN_RECALL_SECONDARY * 100
    )
    & (
        df_cv_summary["Precision Fraud (%) Mean"]
        >= TARGET_MIN_PRECISION_SECONDARY * 100
    )
].copy()

if not primary_eligible.empty:
    cv_eligible = primary_eligible
    cv_selection_status = "PRIMARY_CONSTRAINTS_MET"
elif not secondary_eligible.empty:
    cv_eligible = secondary_eligible
    cv_selection_status = "SECONDARY_CONSTRAINTS_MET"
else:
    cv_eligible = df_cv_summary.copy()
    cv_selection_status = "FALLBACK_ALL_MODELS"

best_cv_row = cv_eligible.sort_values(
    by=[
        "F1 Stability Score",
        "PR-AUC (%) Mean",
        "F1 Fraud (%) Mean",
        "Precision Fraud (%) Mean",
        "Detection Rate (%) Mean",
        "Threshold Std",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        False,
        True,
    ],
).iloc[0]

selected_model_name = best_cv_row["Model"]

print("\n" + "=" * 112)
print("MODEL TERPILIH BERDASARKAN PERFORMA + STABILITAS")
print("=" * 112)
print(f"Status seleksi        : {cv_selection_status}")
print(f"Model                 : {selected_model_name}")
print(
    f"Robust threshold      : "
    f"{best_cv_row['Robust Threshold']:.6f}"
)
print(
    f"F1 CV                 : "
    f"{best_cv_row['F1 Fraud (%) Mean']:.4f} ± "
    f"{best_cv_row['F1 Fraud (%) Std']:.4f}%"
)
print(
    f"F1 stability score    : "
    f"{best_cv_row['F1 Stability Score']:.4f}"
)
print(
    f"Recall CV             : "
    f"{best_cv_row['Detection Rate (%) Mean']:.4f} ± "
    f"{best_cv_row['Detection Rate (%) Std']:.4f}%"
)
print(
    f"Precision CV          : "
    f"{best_cv_row['Precision Fraud (%) Mean']:.4f} ± "
    f"{best_cv_row['Precision Fraud (%) Std']:.4f}%"
)
print(
    f"PR-AUC CV             : "
    f"{best_cv_row['PR-AUC (%) Mean']:.4f} ± "
    f"{best_cv_row['PR-AUC (%) Std']:.4f}%"
)
print(
    f"Worst-fold F1/Recall  : "
    f"{best_cv_row['F1 Fraud (%) Min']:.4f}% / "
    f"{best_cv_row['Detection Rate (%) Min']:.4f}%"
)
print(
    f"Threshold std / IQR   : "
    f"{best_cv_row['Threshold Std']:.6f} / "
    f"{best_cv_row['Threshold IQR']:.6f}"
)

df_cv_summary


In [ ]:
# ============================================================
# VISUALISASI REPEATED 5×5 DAN THRESHOLD STABILITY
# ============================================================

plot_metrics = [
    "Accuracy (%)",
    "Precision Fraud (%)",
    "Detection Rate (%)",
    "F1 Fraud (%)",
    "ROC-AUC (%)",
    "PR-AUC (%)",
]

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

for ax, metric in zip(axes, plot_metrics):
    means = [
        df_cv_summary.loc[
            df_cv_summary["Model"] == model,
            f"{metric} Mean",
        ].iloc[0]
        for model in MODEL_NAMES
    ]
    stds = [
        df_cv_summary.loc[
            df_cv_summary["Model"] == model,
            f"{metric} Std",
        ].iloc[0]
        for model in MODEL_NAMES
    ]
    labels = [
        model.replace("AE + ", "")
        for model in MODEL_NAMES
    ]

    bars = ax.bar(
        labels,
        means,
        yerr=stds,
        capsize=4,
        alpha=0.85,
    )

    for bar, value in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax.set_title(metric, fontweight="bold")
    ax.set_ylabel("Score (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    (
        "Hybrid AE + SMOTE — "
        f"Repeated {N_SPLITS}-Fold × {N_REPEATS}"
    ),
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    "komparasi_repeated_5x5_stable_models.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

# Distribusi 25 threshold cross-fitted.
plt.figure(figsize=(13, 6))
sns.boxplot(
    data=df_crossfit_thresholds,
    x="Model",
    y="Threshold",
)
sns.stripplot(
    data=df_crossfit_thresholds,
    x="Model",
    y="Threshold",
    alpha=0.55,
    size=4,
)
plt.title(
    "Distribusi 25 Cross-Fitted OOF Threshold",
    fontweight="bold",
)
plt.xlabel("Model")
plt.ylabel("Threshold")
plt.xticks(rotation=18)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(
    "crossfit_threshold_stability_25folds.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## Final Training, Validation Confirmation, dan Final Testing

Seluruh model dilatih ulang pada training pool 80%.

- CatBoost final terdiri atas tiga estimator dengan seed berbeda.
- Probabilitas CatBoost dirata-ratakan.
- Setiap model menggunakan fixed threshold berupa median dari 25 cross-fitted OOF thresholds.
- Validation holdout tidak digunakan untuk mengubah threshold; hanya untuk konfirmasi.
- Final testing tidak digunakan untuk memilih model, seed, hyperparameter, maupun threshold.


In [ ]:
# ============================================================
# FINAL TRAINING + FIXED ROBUST OOF THRESHOLD
# ============================================================

print("\n" + "=" * 108)
print("FINAL TRAINING / VALIDATION CONFIRMATION / FINAL TESTING")
print("=" * 108)

final_imputer, final_scaler, X_train_scaled = fit_preprocessor(
    X_train_pool
)
X_val_scaled = transform_preprocessor(
    final_imputer,
    final_scaler,
    X_val_raw,
)
X_test_scaled = transform_preprocessor(
    final_imputer,
    final_scaler,
    X_test_raw,
)

X_train_normal = X_train_scaled[y_train_pool == 0]
final_autoencoder, final_encoder, final_ae_history = (
    fit_autoencoder_on_normal(
        X_train_normal,
        seed=RANDOM_STATE,
    )
)

X_train_hybrid = transform_hybrid_features(
    final_autoencoder,
    final_encoder,
    X_train_scaled,
)
X_val_hybrid = transform_hybrid_features(
    final_autoencoder,
    final_encoder,
    X_val_scaled,
)
X_test_hybrid = transform_hybrid_features(
    final_autoencoder,
    final_encoder,
    X_test_scaled,
)

X_final_train, y_final_train = apply_smote(
    X_train_hybrid,
    y_train_pool,
    label="Final",
    seed=RANDOM_STATE,
)

robust_threshold_map = dict(
    zip(
        df_threshold_summary["Model"],
        df_threshold_summary["Robust Threshold"],
    )
)

final_records = []
final_validation_records = []
final_predictions = {}
trained_final_models = {}

for model_name in MODEL_NAMES:
    model_bundle = build_model_bundle(
        model_name,
        base_seed=RANDOM_STATE,
    )

    print(
        f"\n[Final] Training {model_name} "
        f"({len(model_bundle)} estimator)"
    )

    fit_model_bundle(
        model_bundle,
        X_final_train,
        y_final_train,
    )

    threshold = float(robust_threshold_map[model_name])

    prob_val = predict_model_bundle(
        model_bundle,
        X_val_hybrid,
    )
    pred_val = (prob_val >= threshold).astype(np.int8)

    validation_metrics = compute_binary_metrics(
        y_val_holdout,
        pred_val,
        prob_val,
    )
    validation_status = threshold_confirmation_status(
        validation_metrics
    )

    prob_test = predict_model_bundle(
        model_bundle,
        X_test_hybrid,
    )
    pred_test = (prob_test >= threshold).astype(np.int8)

    test_metrics = compute_binary_metrics(
        y_test_holdout,
        pred_test,
        prob_test,
    )

    final_validation_records.append(
        {
            "Model": model_name,
            "Estimator Count": len(model_bundle),
            "Robust OOF Threshold": threshold,
            "Validation Status": validation_status,
            **validation_metrics,
        }
    )

    final_records.append(
        {
            "Model": model_name,
            "Estimator Count": len(model_bundle),
            "Threshold": threshold,
            "Threshold Source": (
                "MEDIAN_25_CROSSFIT_OOF_THRESHOLDS"
            ),
            "Validation Status": validation_status,
            **test_metrics,
        }
    )

    final_predictions[model_name] = {
        "y_true": y_test_holdout.copy(),
        "y_pred": pred_test,
        "y_prob": prob_test,
    }
    trained_final_models[model_name] = model_bundle

    print(
        f"  threshold={threshold:.6f} "
        f"[{validation_status}] | "
        f"Val Precision="
        f"{validation_metrics['Precision Fraud (%)']:.4f}% | "
        f"Val Recall="
        f"{validation_metrics['Detection Rate (%)']:.4f}% | "
        f"Val F1={validation_metrics['F1 Fraud (%)']:.4f}% | "
        f"Test Acc={test_metrics['Accuracy (%)']:.4f}% | "
        f"Test Precision="
        f"{test_metrics['Precision Fraud (%)']:.4f}% | "
        f"Test Recall="
        f"{test_metrics['Detection Rate (%)']:.4f}% | "
        f"Test F1={test_metrics['F1 Fraud (%)']:.4f}% | "
        f"Test PR-AUC={test_metrics['PR-AUC (%)']:.4f}%"
    )

df_final_validation = pd.DataFrame(
    final_validation_records
)
df_final_summary = pd.DataFrame(final_records)

print("\nVALIDATION CONFIRMATION")
display(df_final_validation)

print("\nFINAL TESTING")
df_final_summary


In [ ]:
# ============================================================
# CLASSIFICATION REPORT FINAL TESTING
# ============================================================

for model_name, payload in final_predictions.items():
    print("\n" + "=" * 88)
    print(f"{model_name} — FINAL TESTING 10%")
    print("=" * 88)
    print(
        classification_report(
            payload["y_true"],
            payload["y_pred"],
            labels=[0, 1],
            target_names=["Transaksi Sah (0)", "Penipuan (1)"],
            digits=4,
            zero_division=0,
        )
    )


In [ ]:
# ============================================================
# HEATMAP FINAL METRICS
# ============================================================

heatmap_columns = [
    "Accuracy (%)",
    "Precision Fraud (%)",
    "Detection Rate (%)",
    "F1 Fraud (%)",
    "ROC-AUC (%)",
    "PR-AUC (%)",
]

heatmap_data = df_final_summary.set_index("Model")[heatmap_columns]

plt.figure(figsize=(12, 4.5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".4f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="gray",
)
plt.title(
    "Perbandingan Performa — Final Testing 10%",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
plt.xticks(rotation=20)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("heatmap_final_testing_4model.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# CONFUSION MATRIX FINAL TESTING
# ============================================================

n_models = len(final_predictions)
fig, axes = plt.subplots(1, n_models, figsize=(5.2 * n_models, 4.8))

if n_models == 1:
    axes = [axes]

for ax, (model_name, payload) in zip(axes, final_predictions.items()):
    cm = confusion_matrix(
        payload["y_true"],
        payload["y_pred"],
        labels=[0, 1],
    )
    tn, fp, fn, tp = cm.ravel()

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Sah", "Fraud"],
        yticklabels=["Sah", "Fraud"],
    )
    ax.set_title(
        f"{model_name}\nTP={tp} | FP={fp} | FN={fn} | TN={tn}",
        fontsize=9,
        fontweight="bold",
    )
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")

fig.suptitle(
    "Confusion Matrix — Final Testing 10%",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("confusion_matrix_final_testing_4model.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# ROC CURVE FINAL TESTING
# ============================================================

plt.figure(figsize=(9, 7))

for model_name, payload in final_predictions.items():
    fpr, tpr, _ = roc_curve(payload["y_true"], payload["y_prob"])
    auc_value = roc_auc_score(payload["y_true"], payload["y_prob"])

    plt.plot(
        fpr,
        tpr,
        lw=2,
        label=f"{model_name} (AUC={auc_value:.4f})",
    )

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title("ROC Curve — Final Testing 10%", fontweight="bold")
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("roc_curve_final_testing_4model.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# PRECISION-RECALL CURVE FINAL TESTING
# ============================================================

plt.figure(figsize=(9, 7))

baseline = y_test_holdout.mean()

for model_name, payload in final_predictions.items():
    precision, recall, _ = precision_recall_curve(
        payload["y_true"],
        payload["y_prob"],
    )
    pr_auc = average_precision_score(
        payload["y_true"],
        payload["y_prob"],
    )

    plt.plot(
        recall,
        precision,
        lw=2,
        label=f"{model_name} (AP={pr_auc:.4f})",
    )

plt.axhline(
    baseline,
    linestyle="--",
    linewidth=1,
    label=f"Fraud prevalence={baseline:.5f}",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Final Testing 10%", fontweight="bold")
plt.legend(loc="best", fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pr_curve_final_testing_4model.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# KESIMPULAN — REPEATED 5×5 + CATBOOST MULTI-SEED
# ============================================================

selected_cv_row = df_cv_summary[
    df_cv_summary["Model"] == selected_model_name
].iloc[0]

selected_validation_row = df_final_validation[
    df_final_validation["Model"] == selected_model_name
].iloc[0]

selected_test_row = df_final_summary[
    df_final_summary["Model"] == selected_model_name
].iloc[0]

print("=" * 116)
print(
    "MODEL REKOMENDASI BERDASARKAN REPEATED CV, "
    "CROSS-FITTED THRESHOLD, DAN STABILITAS"
)
print("=" * 116)

print(f"Model                    : {selected_model_name}")
print(f"Status pemilihan         : {cv_selection_status}")
print(
    f"Robust OOF threshold     : "
    f"{selected_cv_row['Robust Threshold']:.6f}"
)
print(
    f"Threshold std / IQR      : "
    f"{selected_cv_row['Threshold Std']:.6f} / "
    f"{selected_cv_row['Threshold IQR']:.6f}"
)
print(
    f"CV Accuracy              : "
    f"{selected_cv_row['Accuracy (%) Mean']:.4f} ± "
    f"{selected_cv_row['Accuracy (%) Std']:.4f}%"
)
print(
    f"CV Precision Fraud       : "
    f"{selected_cv_row['Precision Fraud (%) Mean']:.4f} ± "
    f"{selected_cv_row['Precision Fraud (%) Std']:.4f}%"
)
print(
    f"CV Detection Rate        : "
    f"{selected_cv_row['Detection Rate (%) Mean']:.4f} ± "
    f"{selected_cv_row['Detection Rate (%) Std']:.4f}%"
)
print(
    f"CV F1 Fraud              : "
    f"{selected_cv_row['F1 Fraud (%) Mean']:.4f} ± "
    f"{selected_cv_row['F1 Fraud (%) Std']:.4f}%"
)
print(
    f"CV PR-AUC                : "
    f"{selected_cv_row['PR-AUC (%) Mean']:.4f} ± "
    f"{selected_cv_row['PR-AUC (%) Std']:.4f}%"
)
print(
    f"F1 stability score       : "
    f"{selected_cv_row['F1 Stability Score']:.4f}"
)
print(
    f"Worst-fold F1 / Recall   : "
    f"{selected_cv_row['F1 Fraud (%) Min']:.4f}% / "
    f"{selected_cv_row['Detection Rate (%) Min']:.4f}%"
)

print("\nVALIDATION CONFIRMATION")
print(
    f"Status                   : "
    f"{selected_validation_row['Validation Status']}"
)
print(
    f"Precision / Recall / F1  : "
    f"{selected_validation_row['Precision Fraud (%)']:.4f}% / "
    f"{selected_validation_row['Detection Rate (%)']:.4f}% / "
    f"{selected_validation_row['F1 Fraud (%)']:.4f}%"
)

print("\nFINAL TESTING — TIDAK DIPAKAI UNTUK TUNING")
for key in [
    "Estimator Count",
    "Threshold",
    "Threshold Source",
    "Validation Status",
    "Accuracy (%)",
    "Precision Fraud (%)",
    "Detection Rate (%)",
    "F1 Fraud (%)",
    "ROC-AUC (%)",
    "PR-AUC (%)",
    "TP",
    "FP",
    "FN",
    "TN",
]:
    value = selected_test_row[key]

    if isinstance(value, (float, np.floating)):
        print(f"{key:<25}: {value:.6f}")
    else:
        print(f"{key:<25}: {value}")

print("\nInterpretasi:")
print("1. Repeated 5×5 memberi 25 estimasi outer-validation.")
print("2. CatBoost multi-seed mereduksi variance probabilitas satu seed.")
print("3. Threshold setiap fold dicari tanpa menggunakan sampel fold evaluasi.")
print("4. Threshold final adalah median dari 25 threshold cross-fitted.")
print("5. Model dipilih berdasarkan performa dan F1 Stability Score.")
print("6. Final testing tidak digunakan untuk tuning atau pemilihan model.")


In [ ]:
# ============================================================
# EXPORT HASIL UNTUK PAPER / AUDIT
# ============================================================

df_ranking_folds.to_csv(
    "repeated_5x5_threshold_independent_ranking_metrics.csv",
    index=False,
)
df_crossfit_thresholds.to_csv(
    "crossfit_oof_thresholds_25folds.csv",
    index=False,
)
df_threshold_summary.to_csv(
    "robust_global_threshold_stability_summary.csv",
    index=False,
)
df_cv_folds.to_csv(
    "crossfit_outer_fold_metrics_5x5.csv",
    index=False,
)
df_cv_summary.to_csv(
    "repeated_5x5_summary_mean_std_ci95.csv",
    index=False,
)
df_global_oof_reference.to_csv(
    "global_averaged_oof_reference_metrics.csv",
    index=False,
)
df_final_validation.to_csv(
    "final_validation_threshold_confirmation.csv",
    index=False,
)
df_final_summary.to_csv(
    "final_testing_summary.csv",
    index=False,
)

# Simpan probabilitas OOF secara terkompresi.
np.savez_compressed(
    "repeated_5x5_oof_probabilities.npz",
    y_true=y_train_pool,
    fold_ids=oof_fold_ids,
    catboost_3seed=oof_probabilities[
        CATBOOST_ENSEMBLE_NAME
    ],
    extratrees=oof_probabilities["AE + ExtraTrees"],
    random_forest=oof_probabilities["AE + Random Forest"],
    lightgbm=oof_probabilities["AE + LightGBM"],
)

selected_model_export = pd.DataFrame(
    [
        {
            "Selected Model": selected_model_name,
            "Selection Source": (
                f"Repeated {N_SPLITS}-Fold × "
                f"{N_REPEATS} with Cross-Fitted OOF Threshold"
            ),
            "Selection Status": cv_selection_status,
            "CatBoost Ensemble Size": (
                CATBOOST_ENSEMBLE_SIZE
                if selected_model_name == CATBOOST_ENSEMBLE_NAME
                else 1
            ),
            "Robust OOF Threshold": (
                selected_cv_row["Robust Threshold"]
            ),
            "Threshold Std": (
                selected_cv_row["Threshold Std"]
            ),
            "Threshold IQR": (
                selected_cv_row["Threshold IQR"]
            ),
            "CV Accuracy Mean (%)": (
                selected_cv_row["Accuracy (%) Mean"]
            ),
            "CV Accuracy Std (%)": (
                selected_cv_row["Accuracy (%) Std"]
            ),
            "CV Precision Mean (%)": (
                selected_cv_row["Precision Fraud (%) Mean"]
            ),
            "CV Precision Std (%)": (
                selected_cv_row["Precision Fraud (%) Std"]
            ),
            "CV Recall Mean (%)": (
                selected_cv_row["Detection Rate (%) Mean"]
            ),
            "CV Recall Std (%)": (
                selected_cv_row["Detection Rate (%) Std"]
            ),
            "CV F1 Mean (%)": (
                selected_cv_row["F1 Fraud (%) Mean"]
            ),
            "CV F1 Std (%)": (
                selected_cv_row["F1 Fraud (%) Std"]
            ),
            "Worst-Fold F1 (%)": (
                selected_cv_row["F1 Fraud (%) Min"]
            ),
            "Worst-Fold Recall (%)": (
                selected_cv_row["Detection Rate (%) Min"]
            ),
            "F1 Stability Score": (
                selected_cv_row["F1 Stability Score"]
            ),
            "CV PR-AUC Mean (%)": (
                selected_cv_row["PR-AUC (%) Mean"]
            ),
            "CV PR-AUC Std (%)": (
                selected_cv_row["PR-AUC (%) Std"]
            ),
            "Validation Confirmation": (
                selected_validation_row["Validation Status"]
            ),
        }
    ]
)

selected_model_export.to_csv(
    "selected_model_repeated_5x5_stable.csv",
    index=False,
)

print("✅ File hasil tersimpan:")
print("- repeated_5x5_threshold_independent_ranking_metrics.csv")
print("- crossfit_oof_thresholds_25folds.csv")
print("- robust_global_threshold_stability_summary.csv")
print("- crossfit_outer_fold_metrics_5x5.csv")
print("- repeated_5x5_summary_mean_std_ci95.csv")
print("- global_averaged_oof_reference_metrics.csv")
print("- final_validation_threshold_confirmation.csv")
print("- final_testing_summary.csv")
print("- repeated_5x5_oof_probabilities.npz")
print("- selected_model_repeated_5x5_stable.csv")
